# HLCE · Aplicación Streamlit de analítica humano–LLM

Notebook autocontenido que genera una aplicación Streamlit inspirada en la infografía metodológica del proyecto.

**Principios:**
- no incluye datos sintéticos ni dataset de demostración;
- todo resultado se calcula sobre el CSV cargado;
- cada método declara sus requisitos;
- las métricas no justificables por el CSV se marcan como no computables;
- permite ejecutar una familia o **TODO lo computable**.

## 1. Dependencias

`sentence-transformers` permite SBERT. En modo `auto`, si SBERT no está disponible, la aplicación usa TF–IDF + SVD calculado exclusivamente sobre el texto cargado.

In [1]:
%pip install -q streamlit pandas numpy scipy scikit-learn plotly networkx sentence-transformers

Note: you may need to restart the kernel to use updated packages.


## 2. Código de la aplicación

Esta celda escribe `hlce_streamlit_app.py`. No descarga ni crea ningún CSV.

In [1]:
from pathlib import Path

APP_CODE = 'from __future__ import annotations\nimport io, math, re, zlib\nfrom itertools import combinations\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nimport plotly.express as px\nimport plotly.graph_objects as go\nimport networkx as nx\nfrom scipy.stats import entropy as sp_entropy, hypergeom, kruskal, mannwhitneyu\nfrom sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer\nfrom sklearn.decomposition import TruncatedSVD, PCA\nfrom sklearn.preprocessing import normalize\n\nst.set_page_config(page_title=\'Mapa Analítico HLCE\', page_icon=\'🧭\', layout=\'wide\')\nNAVY=\'#123B6D\'; BLUE=\'#2673B8\'; CYAN=\'#1AA6A6\'; ORANGE=\'#E66A2C\'; BG=\'#F5FAFD\'\nst.markdown(f\'\'\'<style>\n.stApp{{background:{BG}}}.main .block-container{{max-width:1500px;padding-top:1.2rem}}\n.hero{{background:linear-gradient(110deg,#0F315D,#18568C 60%,#157E89);border-radius:24px;padding:28px 32px;color:white;box-shadow:0 14px 35px rgba(18,59,109,.18)}}\n.hero h1{{color:white;margin:0 0 8px;font-size:2.1rem}}.hero p{{margin:0;opacity:.92}}\n.card{{background:white;border:1px solid rgba(38,115,184,.22);border-top:5px solid {BLUE};border-radius:18px;padding:15px;min-height:210px;box-shadow:0 8px 24px rgba(18,59,109,.07)}}\n.card.teal{{border-top-color:{CYAN}}}.family{{font-size:.78rem;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:{NAVY}}}\n.card h4{{color:{NAVY};margin:.35rem 0}}.card p{{font-size:.9rem;color:#17324D;line-height:1.35}}.formula{{background:#EEF6FB;border:1px solid #D7EAF5;border-radius:10px;padding:8px;color:{NAVY};font-family:serif;font-size:.88rem}}\n.ok,.no{{display:inline-block;margin-top:8px;padding:3px 8px;border-radius:999px;font-size:.72rem;font-weight:700}}.ok{{background:#E8F7F2;color:#136D5D}}.no{{background:#FFF0EA;color:#A84D24}}\ndiv[data-testid=\'stMetric\']{{background:white;border:1px solid #DFEBF3;padding:12px 14px;border-radius:14px}}\n</style>\'\'\', unsafe_allow_html=True)\nst.markdown(\'\'\'<div class="hero"><h1>Mapa unificado de metodologías analíticas</h1><p>Aplicación interactiva para conversaciones humano–LLM. Todas las cifras y gráficos se calculan exclusivamente sobre el CSV cargado. No se generan observaciones, scores ni datos sintéticos.</p></div>\'\'\', unsafe_allow_html=True)\n\nALIASES={\n\'message\':[\'message\',\'text\',\'utterance\',\'content\',\'raw_text\',\'response\'],\n\'actor\':[\'actor\',\'speaker\',\'role\',\'author\',\'participant\'],\n\'condition\':[\'condition\',\'group\',\'regime\',\'experimental_condition\',\'arm\'],\n\'turn\':[\'turn\',\'turn_id\',\'turn_index\',\'step\',\'order\'],\n\'conversation_id\':[\'conversation_id\',\'conversation\',\'dialogue_id\',\'session_id\',\'chat_id\',\'thread_id\'],\n\'timestamp\':[\'timestamp\',\'datetime\',\'time\',\'created_at\'],\n\'claim\':[\'claim\',\'statement\',\'hypothesis\'],\'source\':[\'source\',\'reference\',\'citation\']}\nHLCE={\n\'information_flow\':[\'information_flow\',\'info_flow\'],\'reliability\':[\'reliability\'],\'durability\':[\'durability\',\'persistence\'],\'trust\':[\'trust\'],\n\'procedural_transparency\':[\'procedural_transparency\',\'proc_transparency\'],\'informational_transparency\':[\'informational_transparency\',\'info_transparency\'],\n\'individualization\':[\'individualization\',\'personalization\'],\'transformation\':[\'transformation\']}\nWORD_RE=re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9_\']+",re.UNICODE)\nSTOP={\'de\',\'la\',\'el\',\'y\',\'en\',\'a\',\'los\',\'las\',\'un\',\'una\',\'que\',\'es\',\'por\',\'con\',\'para\',\'del\',\'al\',\'se\',\'lo\',\'como\',\'más\',\'pero\',\'su\',\'sus\',\'le\',\'ya\',\'o\',\'este\',\'esta\',\'esto\',\'si\',\'no\',\'me\',\'te\',\'the\',\'a\',\'an\',\'and\',\'or\',\'of\',\'to\',\'in\',\'is\',\'are\',\'was\',\'were\',\'for\',\'with\',\'on\',\'that\',\'this\',\'it\',\'as\',\'be\',\'by\',\'from\',\'at\',\'i\',\'you\',\'we\',\'they\',\'he\',\'she\',\'not\'}\n\ndef resolve(df,key):\n    low={str(c).lower().strip():c for c in df.columns}\n    for a in ALIASES.get(key,[key]):\n        if a.lower() in low:return low[a.lower()]\n    return None\n\ndef resolve_any(df,aliases):\n    low={str(c).lower().strip():c for c in df.columns}\n    for a in aliases:\n        if a.lower() in low:return low[a.lower()]\n    return None\n\ndef toks(s):return [x.lower() for x in WORD_RE.findall(str(s))]\ndef safe(a,b):return np.nan if not b or pd.isna(b) else a/b\n\ndef normalize_schema(raw):\n    m={k:resolve(raw,k) for k in ALIASES}\n    if not m[\'message\']: raise ValueError(\'Falta una columna textual: message/text/utterance/content/raw_text.\')\n    out=pd.DataFrame(index=raw.index)\n    out[\'message\']=raw[m[\'message\']].fillna(\'\').astype(str)\n    out[\'actor\']=raw[m[\'actor\']].fillna(\'DESCONOCIDO\').astype(str) if m[\'actor\'] else \'DESCONOCIDO\'\n    out[\'condition\']=raw[m[\'condition\']].fillna(\'UNICA\').astype(str) if m[\'condition\'] else \'UNICA\'\n    out[\'conversation_id\']=raw[m[\'conversation_id\']].fillna(\'conv_1\').astype(str) if m[\'conversation_id\'] else \'conv_1\'\n    if m[\'turn\']:\n        out[\'turn\']=pd.to_numeric(raw[m[\'turn\']],errors=\'coerce\')\n        out[\'turn\']=out[\'turn\'].fillna(out.groupby(\'conversation_id\').cumcount()+1)\n    else: out[\'turn\']=out.groupby(\'conversation_id\').cumcount()+1\n    out[\'timestamp\']=pd.to_datetime(raw[m[\'timestamp\']],errors=\'coerce\') if m[\'timestamp\'] else pd.NaT\n    for c in raw.columns:\n        if c not in out.columns: out[c]=raw[c]\n    out[\'_row\']=np.arange(len(out)); out[\'_tokens\']=out[\'message\'].map(toks); out[\'word_count\']=out[\'_tokens\'].map(len)\n    out=out.sort_values([\'conversation_id\',\'turn\',\'_row\']).reset_index(drop=True)\n    out[\'cumulative_words\']=out.groupby(\'conversation_id\')[\'word_count\'].cumsum()\n    return out,m\n\n@st.cache_resource(show_spinner=False)\ndef load_sbert(name):\n    from sentence_transformers import SentenceTransformer\n    return SentenceTransformer(name)\n\n@st.cache_data(show_spinner=False)\ndef embed_cached(text_tuple, backend, model_name):\n    if backend in (\'auto\',\'sbert\'):\n        try:\n            M=load_sbert(model_name); E=M.encode(list(text_tuple),normalize_embeddings=True,show_progress_bar=False)\n            return np.asarray(E,float),f\'SBERT · {model_name}\'\n        except Exception as e:\n            if backend==\'sbert\': raise RuntimeError(f\'No se pudo ejecutar SBERT: {e}\')\n    V=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,max_features=6000,min_df=1)\n    X=V.fit_transform(list(text_tuple)); ns,nf=X.shape\n    if ns>=3 and nf>=3:\n        nc=max(2,min(64,ns-1,nf-1)); E=TruncatedSVD(n_components=nc,random_state=42).fit_transform(X)\n    else:E=X.toarray()\n    return normalize(E), \'TF–IDF + SVD\'\n\ndef getE(df,b,m):return embed_cached(tuple(df[\'message\'].astype(str)),b,m)\n\ndef ttr(x):return safe(len(set(x)),len(x))\ndef mattr(x,w=50):\n    if not x:return np.nan\n    if len(x)<=w:return ttr(x)\n    return float(np.mean([len(set(x[i:i+w]))/w for i in range(len(x)-w+1)]))\ndef mtld(x,thr=.72):\n    if len(x)<10:return np.nan\n    def p(seq):\n        fac=0.; S=set(); n=0\n        for z in seq:\n            n+=1;S.add(z);v=len(S)/n\n            if v<=thr:fac+=1;S=set();n=0\n        if n:fac+=(1-len(S)/n)/max(1e-12,1-thr)\n        return len(seq)/fac if fac else np.nan\n    return float(np.nanmean([p(x),p(list(reversed(x)))]))\ndef hdd(x,s=42):\n    N=len(x)\n    if not N:return np.nan\n    d=min(s,N); f=pd.Series(x).value_counts().values\n    return float(np.sum([hypergeom.sf(0,N,int(v),d) for v in f])/d)\n\ndef lexical_summary(df):\n    rows=[]; keys=[\'condition\']+([\'actor\'] if df[\'actor\'].nunique()>1 else [])\n    for k,g in df.groupby(keys,dropna=False):\n        if not isinstance(k,tuple):k=(k,)\n        x=[t for row in g[\'_tokens\'] for t in row]; r=dict(zip(keys,k))\n        r.update(turns=len(g),tokens=len(x),types=len(set(x)),TTR=ttr(x),MATTR_50=mattr(x),MTLD=mtld(x),HDD_42=hdd(x),mean_words_per_turn=g[\'word_count\'].mean())\n        rows.append(r)\n    return pd.DataFrame(rows)\n\ndef distinctive(df,n=15):\n    if df[\'condition\'].nunique()<2:return pd.DataFrame()\n    g=df.groupby(\'condition\')[\'message\'].apply(lambda s:\'\\n\'.join(s)).sort_index()\n    V=TfidfVectorizer(ngram_range=(1,2),max_features=4000,sublinear_tf=True,min_df=1,stop_words=list(STOP))\n    X=V.fit_transform(g.values); terms=np.array(V.get_feature_names_out()); rows=[]\n    for i,c in enumerate(g.index):\n        sc=X[i].toarray().ravel(); ids=np.argsort(sc)[-n:][::-1]\n        rows += [dict(condition=c,rank=r+1,term=terms[j],tfidf=float(sc[j])) for r,j in enumerate(ids)]\n    return pd.DataFrame(rows)\n\ndef semantic_metrics(df,E):\n    o=df[[\'conversation_id\',\'condition\',\'turn\',\'actor\',\'message\',\'word_count\']].copy(); n=len(o)\n    for c in [\'adjacent_similarity\',\'adjacent_distance\',\'semantic_novelty\',\'prior_other_actor_similarity\']:o[c]=np.nan\n    cents={}\n    for c,idx in o.groupby(\'condition\').groups.items():\n        v=E[list(idx)].mean(0); cents[c]=v/np.linalg.norm(v) if np.linalg.norm(v) else v\n    o[\'condition_centroid_similarity\']=[float(np.dot(E[i],cents[o.loc[i,\'condition\']])) for i in range(n)]\n    for _,idxs in o.groupby(\'conversation_id\').groups.items():\n        ids=list(idxs)\n        for p,i in enumerate(ids):\n            if p:\n                j=ids[p-1]; s=float(np.dot(E[i],E[j])); o.loc[i,\'adjacent_similarity\']=s; o.loc[i,\'adjacent_distance\']=1-s\n                prev=ids[:p]; o.loc[i,\'semantic_novelty\']=1-float(np.max(E[prev]@E[i]))\n                other=[k for k in prev if o.loc[k,\'actor\']!=o.loc[i,\'actor\']]\n                if other:\n                    v=E[other].mean(0); v=v/np.linalg.norm(v) if np.linalg.norm(v) else v\n                    o.loc[i,\'prior_other_actor_similarity\']=float(np.dot(E[i],v))\n    return o\n\ndef directional(tm):\n    rows=[]\n    for conv,g in tm.groupby(\'conversation_id\'):\n        g=g.sort_values(\'turn\').reset_index(drop=True)\n        for i in range(1,len(g)):\n            if g.loc[i-1,\'actor\']!=g.loc[i,\'actor\'] and pd.notna(g.loc[i,\'adjacent_similarity\']):\n                rows.append(dict(conversation_id=conv,condition=g.loc[i,\'condition\'],direction=f"{g.loc[i-1,\'actor\']} → {g.loc[i,\'actor\']}",similarity=g.loc[i,\'adjacent_similarity\']))\n    if not rows:return pd.DataFrame()\n    r=pd.DataFrame(rows)\n    return r.groupby([\'condition\',\'direction\'],as_index=False).agg(mean_similarity=(\'similarity\',\'mean\'),median_similarity=(\'similarity\',\'median\'),n=(\'similarity\',\'size\'))\n\ndef matched(df,E):\n    if df[\'condition\'].nunique()<2:return pd.DataFrame()\n    rows=[]\n    for key,g in df.groupby([\'turn\',\'actor\']):\n        cs=list(g[\'condition\'].unique())\n        for a,b in combinations(cs,2):\n            ia=g.index[g[\'condition\']==a].tolist(); ib=g.index[g[\'condition\']==b].tolist()\n            if ia and ib:\n                va=normalize(E[ia].mean(0).reshape(1,-1))[0]; vb=normalize(E[ib].mean(0).reshape(1,-1))[0]\n                rows.append(dict(turn=key[0],actor=key[1],condition_A=a,condition_B=b,cosine_similarity=float(np.dot(va,vb))))\n    return pd.DataFrame(rows)\n\ndef info_summary(df):\n    rows=[]\n    for c,g in df.groupby(\'condition\'):\n        x=[t for row in g[\'_tokens\'] for t in row]; txt=\'\\n\'.join(g[\'message\'].astype(str)); counts=pd.Series(x).value_counts().values.astype(float) if x else np.array([])\n        H=float(sp_entropy(counts/counts.sum(),base=2)) if len(counts) else np.nan\n        b=txt.encode(\'utf-8\',errors=\'ignore\'); cr=len(zlib.compress(b,9))/len(b) if b else np.nan\n        rows.append(dict(condition=c,turns=len(g),tokens=len(x),shannon_entropy_bits=H,compression_ratio_zlib=cr,lexical_redundancy=1-ttr(x) if x else np.nan))\n    return pd.DataFrame(rows)\n\ndef divergences(df):\n    if df[\'condition\'].nunique()<2:return pd.DataFrame()\n    g=df.groupby(\'condition\')[\'message\'].apply(lambda s:\'\\n\'.join(s)).sort_index(); V=CountVectorizer(max_features=3000,stop_words=list(STOP)); X=V.fit_transform(g.values).toarray().astype(float)+1e-9; P=X/X.sum(1,keepdims=True); rows=[]\n    for i,j in combinations(range(len(g)),2):\n        p,q=P[i],P[j]; m=.5*(p+q)\n        rows.append(dict(condition_A=g.index[i],condition_B=g.index[j],KL_A_B_bits=float(np.sum(p*np.log2(p/q))),KL_B_A_bits=float(np.sum(q*np.log2(q/p))),JS_divergence_bits=float(.5*np.sum(p*np.log2(p/m))+.5*np.sum(q*np.log2(q/m)))))\n    return pd.DataFrame(rows)\n\ndef sem_graph(df,E,thr=.65,max_nodes=250):\n    n=min(len(df),max_nodes); S=E[:n]@E[:n].T; G=nx.Graph()\n    for i in range(n):G.add_node(i,actor=str(df.loc[i,\'actor\']),condition=str(df.loc[i,\'condition\']),turn=float(df.loc[i,\'turn\']))\n    for i in range(n):\n        for j in range(i+1,n):\n            if S[i,j]>=thr:G.add_edge(i,j,weight=float(S[i,j]))\n    deg=nx.degree_centrality(G); bet=nx.betweenness_centrality(G) if G.number_of_edges() else {i:0 for i in G}; comm=[]; Q=np.nan\n    if G.number_of_edges():\n        try:comm=list(nx.algorithms.community.greedy_modularity_communities(G,weight=\'weight\'));Q=nx.algorithms.community.modularity(G,comm,weight=\'weight\')\n        except Exception:pass\n    cmap={n:k for k,C in enumerate(comm) for n in C}\n    M=pd.DataFrame(dict(node=list(G.nodes),turn=[G.nodes[i][\'turn\'] for i in G],actor=[G.nodes[i][\'actor\'] for i in G],condition=[G.nodes[i][\'condition\'] for i in G],degree_centrality=[deg.get(i,0) for i in G],betweenness=[bet.get(i,0) for i in G],community=[cmap.get(i,-1) for i in G]))\n    return G,M,Q\n\ndef graph_fig(G,M):\n    pos=nx.spring_layout(G,seed=42,weight=\'weight\'); ex=[];ey=[]\n    for u,v in G.edges():\n        x0,y0=pos[u];x1,y1=pos[v];ex += [x0,x1,None];ey += [y0,y1,None]\n    edge=go.Scatter(x=ex,y=ey,mode=\'lines\',line=dict(width=.6,color=\'rgba(90,110,130,.30)\'),hoverinfo=\'none\')\n    nxv=[];nyv=[];hover=[];sizes=[]\n    for n in G.nodes():\n        x,y=pos[n];nxv.append(x);nyv.append(y);r=M[M.node==n].iloc[0];hover.append(f"turn={r.turn}<br>actor={r.actor}<br>condition={r.condition}<br>degree={r.degree_centrality:.3f}");sizes.append(8+26*r.degree_centrality)\n    node=go.Scatter(x=nxv,y=nyv,mode=\'markers\',marker=dict(size=sizes,color=M[\'community\'],colorscale=\'Tealgrn\',showscale=True,line=dict(width=.7,color=\'white\')),text=hover,hoverinfo=\'text\')\n    fig=go.Figure([edge,node]);fig.update_layout(height=620,title=\'Grafo de similitud semántica\',showlegend=False,xaxis=dict(visible=False),yaxis=dict(visible=False),plot_bgcolor=\'white\',margin=dict(l=10,r=10,t=40,b=10));return fig\n\ndef cohend(x,y):\n    x=pd.Series(x).dropna().astype(float).values;y=pd.Series(y).dropna().astype(float).values\n    if len(x)<2 or len(y)<2:return np.nan\n    vx,vy=np.var(x,ddof=1),np.var(y,ddof=1);sp=math.sqrt(((len(x)-1)*vx+(len(y)-1)*vy)/max(1,len(x)+len(y)-2));return (np.mean(x)-np.mean(y))/sp if sp else np.nan\n\ndef cliff(x,y):\n    x=pd.Series(x).dropna().astype(float).values;y=pd.Series(y).dropna().astype(float).values\n    if not len(x) or not len(y):return np.nan\n    return (sum(a>b for a in x for b in y)-sum(a<b for a in x for b in y))/(len(x)*len(y))\n\ndef compare(df,metric):\n    gs={str(c):g[metric].dropna().astype(float).values for c,g in df.groupby(\'condition\') if g[metric].dropna().shape[0]}\n    omni=[];pairs=[]\n    if len(gs)>=2:\n        try:H,p=kruskal(*gs.values());omni=[dict(test=\'Kruskal–Wallis\',statistic=H,p_value=p)]\n        except Exception:pass\n    for a,b in combinations(gs,2):\n        x,y=gs[a],gs[b]\n        try:U,p=mannwhitneyu(x,y,alternative=\'two-sided\')\n        except Exception:U,p=np.nan,np.nan\n        pairs.append(dict(condition_A=a,condition_B=b,mean_A=np.mean(x),mean_B=np.mean(y),delta_mean=np.mean(x)-np.mean(y),cohens_d=cohend(x,y),cliffs_delta=cliff(x,y),mann_whitney_U=U,p_value=p))\n    return pd.DataFrame(omni),pd.DataFrame(pairs)\n\ndef dl(df,label,name):st.download_button(label,df.to_csv(index=False).encode(\'utf-8\'),file_name=name,mime=\'text/csv\',use_container_width=True)\nMETHODS=[\n(\'structure\',\'1 · Preparación y estructuración\',\'Integridad, esquema y turn-taking\',\'Valida columnas, vacíos, secuencia de turnos, actores, condiciones y trazabilidad conversacional.\',\'D = {(cᵢ, tᵢ, aᵢ, uᵢ)}ᵢ₌₁ᴺ\',\'blue\'),\n(\'literature\',\'2 · Revisión conceptual\',\'Mapa claim ↔ fuente\',\'Solo se ejecuta si el CSV contiene claims y referencias explícitas; no infiere bibliografía ausente.\',\'Aᵢⱼ = 1[claimᵢ ↔ sourceⱼ]\',\'teal\'),\n(\'lexical\',\'3 · Análisis léxico-lingüístico\',\'Diversidad, longitud y rasgos textuales\',\'TTR, MATTR, MTLD, HD-D, n-gramas, términos distintivos y distribución de longitud.\',\'TTR = V/N ; MATTR = (1/W) Σ TTR_w\',\'blue\'),\n(\'semantic\',\'4 · Semántica computacional\',\'Similitud, novedad y convergencia\',\'Embeddings del texto, similitud coseno, distancia adyacente, novedad y proximidad direccional.\',\'sim(x,y)= eₓ·eᵧ / (||eₓ|| ||eᵧ||)\',\'teal\'),\n(\'geometry\',\'5 · Geometría del significado\',\'PCA, centroides y trayectorias\',\'Proyección semántica, centroides por actor/condición, trayectorias y turnos emparejados.\',\'Z = XW ; dᵢⱼ = ||zᵢ-zⱼ||₂\',\'blue\'),\n(\'graphs\',\'6 · Estructura relacional y grafos\',\'Red semántica, centralidad y comunidades\',\'Grafo de similitud entre turnos, degree/betweenness, comunidades y modularidad.\',\'Q = (1/2m) Σᵢⱼ(Aᵢⱼ − kᵢkⱼ/2m)δ(cᵢ,cⱼ)\',\'teal\'),\n(\'dynamics\',\'7 · Dinámica conversacional\',\'Ritmo, asimetrías y respuesta\',\'Longitud por turno, acumulación, transición de actores, proximidad HUMAN→AI / AI→HUMAN y latencia si existe.\',\'C = (1/(T−1)) Σₜ₌₂ᵀ sim(uₜ,uₜ₋₁)\',\'blue\'),\n(\'information\',\'8 · Información y complejidad\',\'Entropía, divergencia y compresión\',\'Shannon, KL/JS entre condiciones, ratio zlib y redundancia léxica.\',\'H(X)=−Σpᵢlog₂pᵢ ; D_KL(P||Q)=Σpᵢlog₂(pᵢ/qᵢ)\',\'teal\'),\n(\'coupling\',\'9 · Acoplamiento humano–IA\',\'HLCE / dimensiones observadas\',\'Agrega las ocho dimensiones solo si existen como medidas explícitas en el CSV; no las estima desde texto.\',\'HLCE = Σₖ wₖ zₖ  (solo con zₖ observados)\',\'blue\'),\n(\'experiment\',\'10 · Diseño experimental\',\'Comparación entre condiciones\',\'Contrasta métricas mediante Δ, tamaños de efecto y pruebas no paramétricas.\',\'Δ = x̄_A − x̄_B ; d = (x̄_A−x̄_B)/s_p\',\'teal\')]\n\ndef status(df,id):\n    if id in {\'structure\',\'lexical\',\'semantic\',\'geometry\',\'graphs\',\'dynamics\',\'information\'}:return True,\'COMPUTABLE\'\n    if id==\'literature\':\n        ok=resolve(df,\'claim\') is not None and resolve(df,\'source\') is not None;return ok,\'COMPUTABLE\' if ok else \'Requiere claim + source\'\n    if id==\'coupling\':\n        ok=all(resolve_any(df,a) is not None for a in HLCE.values());return ok,\'COMPUTABLE\' if ok else \'Requiere 8 dimensiones HLCE\'\n    if id==\'experiment\':\n        ok=df[\'condition\'].nunique()>=2;return ok,\'COMPUTABLE\' if ok else \'Requiere ≥2 condiciones\'\n    return False,\'NO DISPONIBLE\'\n\ndef render_structure(df,m,b,model):\n    st.subheader(\'1. Preparación y estructuración\');st.latex(r\'D=\\{(c_i,t_i,a_i,u_i)\\}_{i=1}^{N}\')\n    cs=st.columns(4);cs[0].metric(\'Filas\',len(df));cs[1].metric(\'Conversaciones\',df.conversation_id.nunique());cs[2].metric(\'Actores\',df.actor.nunique());cs[3].metric(\'Condiciones\',df.condition.nunique())\n    diag=pd.DataFrame({\'campo\':[\'message\',\'actor\',\'condition\',\'turn\',\'conversation_id\',\'timestamp\'],\'columna_origen\':[m.get(x) for x in [\'message\',\'actor\',\'condition\',\'turn\',\'conversation_id\',\'timestamp\']]});st.dataframe(diag,use_container_width=True,hide_index=True)\n    dup=df.duplicated([\'conversation_id\',\'turn\',\'actor\',\'message\']).sum();st.info(f\'Duplicados exactos conversación-turno-actor-texto: **{dup}**\')\n    st.dataframe(df[[\'conversation_id\',\'condition\',\'turn\',\'actor\',\'message\',\'word_count\',\'cumulative_words\']].head(120),use_container_width=True,hide_index=True);dl(df.drop(columns=[\'_tokens\'],errors=\'ignore\'),\'Descargar datos normalizados\',\'normalized_conversations.csv\')\n\ndef render_lit(df,m,b,model):\n    st.subheader(\'2. Revisión conceptual y bibliográfica\');st.latex(r\'A_{ij}=\\mathbf{1}[claim_i\\leftrightarrow source_j]\')\n    c,s=resolve(df,\'claim\'),resolve(df,\'source\')\n    if not c or not s:st.warning(\'No computable: faltan claim y source/reference.\');return\n    L=df[[c,s]].dropna().astype(str).drop_duplicates();st.metric(\'Enlaces claim ↔ source\',len(L));st.dataframe(L.head(250),use_container_width=True,hide_index=True)\n    C=L.groupby(c)[s].nunique().sort_values(ascending=False).reset_index(name=\'n_sources\');st.plotly_chart(px.bar(C.head(30),x=\'n_sources\',y=c,orientation=\'h\',title=\'Fuentes por claim\'),use_container_width=True)\n\ndef render_lex(df,m,b,model):\n    st.subheader(\'3. Análisis léxico-lingüístico\');st.latex(r\'\\mathrm{TTR}=V/N,\\qquad \\mathrm{MATTR}=\\frac{1}{W}\\sum_w \\mathrm{TTR}_w\')\n    S=lexical_summary(df);st.dataframe(S,use_container_width=True,hide_index=True);dl(S,\'Descargar métricas léxicas\',\'lexical_metrics.csv\')\n    st.plotly_chart(px.box(df,x=\'condition\',y=\'word_count\',color=\'actor\',points=\'outliers\',title=\'Longitud de mensaje por condición y actor\'),use_container_width=True)\n    T=distinctive(df)\n    if not T.empty:\n        st.plotly_chart(px.bar(T,x=\'tfidf\',y=\'term\',color=\'condition\',facet_col=\'condition\',facet_col_wrap=2,orientation=\'h\',title=\'Términos y n-gramas distintivos (TF–IDF)\'),use_container_width=True);dl(T,\'Descargar términos distintivos\',\'distinctive_lexical_features.csv\')\n\ndef render_sem(df,m,b,model):\n    st.subheader(\'4. Semántica computacional\');st.latex(r\'\\mathrm{sim}(x,y)=\\frac{e_x\\cdot e_y}{\\|e_x\\|\\|e_y\\|}\')\n    E,used=getE(df,b,model);T=semantic_metrics(df,E);st.success(f\'Backend: {used}\')\n    cs=st.columns(3);cs[0].metric(\'Sim. adyacente media\',f"{T.adjacent_similarity.mean():.3f}");cs[1].metric(\'Novedad media\',f"{T.semantic_novelty.mean():.3f}");cs[2].metric(\'Sim. centroide condición\',f"{T.condition_centroid_similarity.mean():.3f}")\n    st.plotly_chart(px.line(T,x=\'turn\',y=\'adjacent_distance\',color=\'condition\',line_group=\'conversation_id\',markers=True,title=\'Distancia semántica adyacente\'),use_container_width=True)\n    st.plotly_chart(px.line(T,x=\'turn\',y=\'semantic_novelty\',color=\'condition\',line_group=\'conversation_id\',markers=True,title=\'Novedad semántica por turno\'),use_container_width=True)\n    D=directional(T)\n    if not D.empty:st.dataframe(D,use_container_width=True,hide_index=True);st.plotly_chart(px.bar(D,x=\'direction\',y=\'mean_similarity\',color=\'condition\',barmode=\'group\',title=\'Proximidad direccional\'),use_container_width=True);dl(D,\'Descargar proximidad direccional\',\'directional_response_proximity.csv\')\n    dl(T,\'Descargar métricas por turno\',\'turn_level_semantic_metrics.csv\')\n\ndef render_geom(df,m,b,model):\n    st.subheader(\'5. Geometría del significado\');st.latex(r\'Z=XW,\\qquad d_{ij}=\\|z_i-z_j\\|_2\')\n    E,used=getE(df,b,model);nc=min(2,E.shape[0],E.shape[1]);Z=PCA(n_components=nc,random_state=42).fit_transform(E) if len(E)>=2 else np.zeros((len(E),1));Z=np.column_stack([Z[:,0],np.zeros(len(Z))]) if Z.shape[1]==1 else Z\n    P=df[[\'condition\',\'actor\',\'turn\',\'conversation_id\',\'message\']].copy();P[\'PC1\']=Z[:,0];P[\'PC2\']=Z[:,1];st.caption(f\'Entrada semántica: {used}. PCA solo proyecta para visualización.\')\n    st.plotly_chart(px.scatter(P,x=\'PC1\',y=\'PC2\',color=\'condition\',symbol=\'actor\',hover_data=[\'turn\',\'conversation_id\'],title=\'Espacio semántico PCA\'),use_container_width=True)\n    st.plotly_chart(px.line(P.sort_values([\'conversation_id\',\'turn\']),x=\'PC1\',y=\'PC2\',color=\'condition\',line_group=\'conversation_id\',markers=True,hover_data=[\'turn\',\'actor\'],title=\'Trayectorias semánticas\'),use_container_width=True)\n    C=P.groupby([\'condition\',\'actor\'],as_index=False).agg(PC1=(\'PC1\',\'mean\'),PC2=(\'PC2\',\'mean\'),n=(\'turn\',\'size\'));st.dataframe(C,use_container_width=True,hide_index=True)\n    M=matched(df,E)\n    if not M.empty:st.dataframe(M.head(300),use_container_width=True,hide_index=True);dl(M,\'Descargar matched turns\',\'matched_turn_cross_condition_similarity.csv\')\n    dl(P,\'Descargar PCA\',\'semantic_geometry_pca.csv\')\ndef render_graph(df,m,b,model):\n    st.subheader(\'6. Estructura relacional y grafos\');st.latex(r\'Q=\\frac{1}{2m}\\sum_{ij}(A_{ij}-\\frac{k_i k_j}{2m})\\delta(c_i,c_j)\')\n    thr=st.slider(\'Umbral de similitud\',.20,.95,.65,.05,key=\'thr\');mx=st.slider(\'Máximo de turnos\',50,500,250,50,key=\'mx\')\n    E,used=getE(df,b,model);G,M,Q=sem_graph(df,E,thr,mx);cs=st.columns(3);cs[0].metric(\'Nodos\',G.number_of_nodes());cs[1].metric(\'Aristas\',G.number_of_edges());cs[2].metric(\'Modularidad\',\'n/d\' if pd.isna(Q) else f\'{Q:.3f}\')\n    st.caption(f\'Backend: {used}\');st.plotly_chart(graph_fig(G,M),use_container_width=True);st.dataframe(M.sort_values(\'degree_centrality\',ascending=False).head(60),use_container_width=True,hide_index=True);dl(M,\'Descargar métricas del grafo\',\'semantic_graph_metrics.csv\')\n\ndef render_dyn(df,m,b,model):\n    st.subheader(\'7. Dinámica conversacional\');st.latex(r\'C=\\frac{1}{T-1}\\sum_{t=2}^{T}\\mathrm{sim}(u_t,u_{t-1})\')\n    st.plotly_chart(px.line(df,x=\'turn\',y=\'word_count\',color=\'actor\',line_group=\'conversation_id\',markers=True,title=\'Longitud por turno\'),use_container_width=True)\n    st.plotly_chart(px.line(df,x=\'turn\',y=\'cumulative_words\',color=\'actor\',line_group=\'conversation_id\',title=\'Palabras acumuladas\'),use_container_width=True)\n    rows=[]\n    for conv,g in df.groupby(\'conversation_id\'):\n        g=g.sort_values(\'turn\').reset_index(drop=True)\n        for i in range(1,len(g)):rows.append(dict(conversation_id=conv,condition=g.loc[i,\'condition\'],from_actor=g.loc[i-1,\'actor\'],to_actor=g.loc[i,\'actor\']))\n    if rows:\n        R=pd.DataFrame(rows).groupby([\'condition\',\'from_actor\',\'to_actor\'],as_index=False).size();st.dataframe(R,use_container_width=True,hide_index=True)\n    E,used=getE(df,b,model);D=directional(semantic_metrics(df,E))\n    if not D.empty:st.markdown(\'**Proximidad semántica direccional**\');st.dataframe(D,use_container_width=True,hide_index=True)\n    if df.timestamp.notna().sum()>=2:\n        L=df[[\'conversation_id\',\'condition\',\'turn\',\'actor\',\'timestamp\']].copy();L[\'latency_s\']=L.groupby(\'conversation_id\').timestamp.diff().dt.total_seconds();st.plotly_chart(px.box(L.dropna(subset=[\'latency_s\']),x=\'condition\',y=\'latency_s\',color=\'actor\',title=\'Latencia observada\'),use_container_width=True)\n    else:st.caption(\'Latencia no calculada: no hay timestamps suficientes.\')\n\ndef render_info(df,m,b,model):\n    st.subheader(\'8. Información y complejidad\');st.latex(r\'H(X)=-\\sum_i p_i\\log_2p_i,\\qquad D_{KL}(P\\|Q)=\\sum_i p_i\\log_2\\frac{p_i}{q_i}\')\n    I=info_summary(df);st.dataframe(I,use_container_width=True,hide_index=True);st.plotly_chart(px.bar(I,x=\'condition\',y=[\'shannon_entropy_bits\',\'compression_ratio_zlib\'],barmode=\'group\',title=\'Entropía y compresión\'),use_container_width=True)\n    D=divergences(df)\n    if not D.empty:st.dataframe(D,use_container_width=True,hide_index=True);dl(D,\'Descargar KL/JS\',\'condition_kl_js_divergence.csv\')\n    dl(I,\'Descargar información y complejidad\',\'condition_information_complexity.csv\')\n\ndef render_coupling(df,m,b,model):\n    st.subheader(\'9. Psicometría y acoplamiento humano–IA\');st.latex(r\'\\mathrm{HLCE}=\\sum_{k=1}^{8}w_k z_k\')\n    cols={d:resolve_any(df,a) for d,a in HLCE.items()};miss=[d for d,c in cols.items() if c is None]\n    if miss:st.warning(\'No se calcula HLCE. Faltan medidas explícitas: \'+\', \'.join(miss)+\'. No se estiman desde texto.\');return\n    W=pd.DataFrame({\'condition\':df.condition})\n    for d,c in cols.items():W[d]=pd.to_numeric(df[c],errors=\'coerce\')\n    Z=W[list(HLCE)].copy()\n    for c in Z:\n        mu=Z[c].mean();sd=Z[c].std(ddof=0);Z[c]=(Z[c]-mu)/sd if sd and not np.isnan(sd) else np.nan\n    cc=st.columns(4);weights={}\n    for i,d in enumerate(HLCE):weights[d]=cc[i%4].number_input(d.replace(\'_\',\' \'),0.0,1.0,1/8,.01,key=\'w_\'+d)\n    s=sum(weights.values())\n    if s<=0:st.error(\'La suma de pesos debe ser > 0.\');return\n    weights={k:v/s for k,v in weights.items()};score=pd.Series(0.,index=W.index);valid=pd.Series(True,index=W.index)\n    for d in HLCE:valid &= Z[d].notna();score += weights[d]*Z[d].fillna(0)\n    W[\'HLCE_descriptivo\']=score.where(valid,np.nan);S=W.groupby(\'condition\',as_index=False)[list(HLCE)+[\'HLCE_descriptivo\']].mean();st.dataframe(S,use_container_width=True,hide_index=True);st.plotly_chart(px.bar(S,x=\'condition\',y=list(HLCE),barmode=\'group\',title=\'Dimensiones HLCE observadas\'),use_container_width=True);dl(W,\'Descargar HLCE observado\',\'hlce_observed_dimensions.csv\')\n    st.info(\'El agregado es descriptivo; no presupone validación psicométrica del índice.\')\n\ndef render_exp(df,m,b,model):\n    st.subheader(\'10. Diseño experimental y comparación de condiciones\');st.latex(r\'\\Delta=\\bar{x}_A-\\bar{x}_B,\\qquad d=\\frac{\\bar{x}_A-\\bar{x}_B}{s_p}\')\n    if df.condition.nunique()<2:st.warning(\'Se requieren al menos dos condiciones.\');return\n    E,used=getE(df,b,model);T=semantic_metrics(df,E);T[\'word_count\']=df.word_count.values\n    opts={\'Longitud del turno\':\'word_count\',\'Similitud adyacente\':\'adjacent_similarity\',\'Distancia adyacente\':\'adjacent_distance\',\'Novedad semántica\':\'semantic_novelty\',\'Sim. centroide condición\':\'condition_centroid_similarity\',\'Proximidad a actor previo\':\'prior_other_actor_similarity\'}\n    lab=st.selectbox(\'Métrica para contrastar\',list(opts));metric=opts[lab];O,P=compare(T[[\'condition\',metric]],metric)\n    if not O.empty:st.dataframe(O,use_container_width=True,hide_index=True)\n    if not P.empty:st.dataframe(P,use_container_width=True,hide_index=True);dl(P,\'Descargar comparación\',\'condition_comparison.csv\')\n    st.plotly_chart(px.box(T,x=\'condition\',y=metric,points=\'outliers\',title=f\'{lab} por condición\'),use_container_width=True)\n\nRENDER={\'structure\':render_structure,\'literature\':render_lit,\'lexical\':render_lex,\'semantic\':render_sem,\'geometry\':render_geom,\'graphs\':render_graph,\'dynamics\':render_dyn,\'information\':render_info,\'coupling\':render_coupling,\'experiment\':render_exp}\n\nwith st.sidebar:\n    st.header(\'Datos y motor semántico\');uploaded=st.file_uploader(\'Cargar CSV\',type=[\'csv\']);sep=st.selectbox(\'Separador\',[\'Auto\',\',\',\';\',\'\\\\t\',\'|\']);backend=st.selectbox(\'Embeddings\',[\'auto\',\'sbert\',\'tfidf-svd\']);model=st.text_input(\'Modelo SBERT\',\'sentence-transformers/all-MiniLM-L6-v2\');st.markdown(\'---\');st.caption(\'Sin dataset de demostración. Sin imputación de contenido. Si AUTO no dispone de SBERT, TF–IDF + SVD se calcula sobre el propio CSV.\')\nif uploaded is None:st.info(\'Carga un CSV. Como mínimo debe incluir message/text/utterance/content/raw_text.\');st.stop()\ntry:\n    rawb=uploaded.getvalue();raw=pd.read_csv(io.BytesIO(rawb),sep=None,engine=\'python\') if sep==\'Auto\' else pd.read_csv(io.BytesIO(rawb),sep=\'\\t\' if sep==\'\\\\t\' else sep);df,mapping=normalize_schema(raw)\nexcept Exception as e:st.error(f\'No se pudo cargar/normalizar el CSV: {e}\');st.stop()\ncs=st.columns(4);cs[0].metric(\'Turnos\',len(df));cs[1].metric(\'Conversaciones\',df.conversation_id.nunique());cs[2].metric(\'Actores\',df.actor.nunique());cs[3].metric(\'Condiciones\',df.condition.nunique())\nst.markdown(\'## Mapa de métodos\');st.caption(\'Pulsa una tarjeta para calcular ese bloque o ejecuta todos los métodos compatibles con el CSV.\')\nif \'sel\' not in st.session_state:st.session_state.sel=None\nif \'all\' not in st.session_state:st.session_state.all=False\nca,cb=st.columns(2)\nif ca.button(\'▶ Ejecutar TODO lo computable\',type=\'primary\',use_container_width=True):st.session_state.all=True;st.session_state.sel=None\nif cb.button(\'Limpiar resultados\',use_container_width=True):st.session_state.all=False;st.session_state.sel=None\nfor i in range(0,len(METHODS),2):\n    cols=st.columns(2)\n    for j,M in enumerate(METHODS[i:i+2]):\n        id,g,t,d,f,tone=M;ok,msg=status(df,id)\n        with cols[j]:\n            st.markdown(f\'\'\'<div class="card {\'teal\' if tone==\'teal\' else \'\'}"><div class="family">{g}</div><h4>{t}</h4><p>{d}</p><div class="formula">{f}</div><span class="{\'ok\' if ok else \'no\'}">{msg}</span></div>\'\'\',unsafe_allow_html=True)\n            if st.button(\'Calcular · \'+t,key=\'run_\'+id,use_container_width=True,disabled=not ok):st.session_state.sel=id;st.session_state.all=False\nst.markdown(\'---\')\nif st.session_state.all:\n    st.markdown(\'## Resultados · ejecución completa\')\n    for M in METHODS:\n        id,g,t,*_=M;ok,msg=status(df,id)\n        with st.expander(f\'{g} · {t}\',expanded=ok):\n            if ok:\n                try:RENDER[id](df,mapping,backend,model)\n                except Exception as e:st.error(f\'No pudo completarse: {e}\')\n            else:st.warning(msg)\nelif st.session_state.sel:\n    id=st.session_state.sel;t=next(x[2] for x in METHODS if x[0]==id);st.markdown(\'## Resultado · \'+t)\n    try:RENDER[id](df,mapping,backend,model)\n    except Exception as e:st.error(f\'No pudo completarse: {e}\')\nelse:st.info(\'Selecciona una metodología del mapa para comenzar.\')\n'

app_file=Path('hlce_streamlit_app.py')
app_file.write_text(APP_CODE,encoding='utf-8')
print('Aplicación escrita en:',app_file.resolve())
print('Líneas:',len(APP_CODE.splitlines()))

Aplicación escrita en: C:\Users\Biempi01\Desktop\Experimento HLCE\hlce_streamlit_app.py
Líneas: 406


## 3. Validación estática

In [2]:
import py_compile
py_compile.compile('hlce_streamlit_app.py',doraise=True)
print('✓ Sintaxis Python válida')

✓ Sintaxis Python válida


## 4. Lanzar Streamlit

Desde una terminal en el mismo directorio:

```bash
streamlit run hlce_streamlit_app.py
```

O desde la celda siguiente mediante Jupyter/Anaconda.

In [ ]:
!streamlit run hlce_streamlit_app.py

## 5. Esquema de CSV

**Mínimo:** `message`, `text`, `utterance`, `content` o `raw_text`.

**Recomendado para HLCE raw:** `conversation_id`, `condition`, `turn`/`turn_id`, `actor`, `message`.

**Opcionales:**
- `timestamp` → latencia real entre turnos;
- `claim` + `source` → mapa claim↔fuente;
- `information_flow`, `reliability`, `durability`, `trust`, `procedural_transparency`, `informational_transparency`, `individualization`, `transformation` → HLCE descriptivo;
- columnas de performance/corrección si el experimento las contiene.

No se imputan dimensiones ausentes ni se estiman desde el texto.

## 6. Familias de análisis incluidas

1. Preparación y estructuración.
2. Revisión conceptual/bibliográfica cuando existen claims y fuentes.
3. Léxico-lingüística: TTR, MATTR, MTLD, HD-D, n-gramas y términos distintivos.
4. Semántica computacional: embeddings, similitud, distancia, novedad y proximidad direccional.
5. Geometría: PCA, centroides, trayectorias y matched-turn cross-condition similarity.
6. Grafos: red semántica, centralidad, comunidades y modularidad.
7. Dinámica conversacional: longitud, acumulación, alternancia, proximidad de respuesta y latencia observada.
8. Información/complejidad: Shannon, KL, JS, zlib y redundancia.
9. Acoplamiento HLCE solo con las ocho dimensiones medidas.
10. Diseño experimental: Δ, Cohen d, Cliff δ, Kruskal–Wallis y Mann–Whitney.